# ADS v2: parity with the current event stream

Compares ads_v2.ScvEventStream with ods.scvepisodeeventstreamfinal for the build window, per EventTypeId: events and distinct event keys on each side, keys in both at the same minute / same day, keys only in one side, and SCV agreement on the same-minute matches (the current stream's negative keys count as unlinked). Writes ads_v2.EventStreamParity. Skipped (with a message) where the current stream is not present.

Generated by `ads_build/v2/build_ads.py` from `families/*.yaml`. Event descriptions come from `seeds/ChillEpisodeTypes.csv`.

**If a cell fails**

1. Read the failing cell's header: what it builds, what it reads, its grain, key and dedupe rule.
2. Fix the cause. Most failures are a source change (a renamed or missing column) or a broken key (duplicates).
   The checks cell at the end names the table and the likely cause.
3. Re-run from the failed cell: every cell rebuilds its table completely (`create or replace`), so re-running is
   always safe. After a session restart, run the first three cells again first (parameters, setup, start log).
4. **Copy the fix back into the config** (`ads_build/v2/families/*.yaml`) and regenerate. This notebook is generated:
   a fix made only here is lost at the next generation (the drift check lists such cells).


In [ ]:
%%pyspark
# Parameters (leave blank when running by hand)
batch_id = ""


In [ ]:
%%pyspark
%run ADS_v2_00_Setup

In [ ]:
%%pyspark
ads_run_log("22_Parity", "started")


In [ ]:
%%pyspark
# ads_v2.EventStreamParity: Per EventTypeId counts, matches and SCV agreement between the current stream and ads_v2.
#
# Reads: ads_v2.ScvEventStream
old_table = globals().get("parity_source") or "ods.scvepisodeeventstreamfinal"
if not spark.catalog.tableExists(old_table):
    print(f"{old_table} not found: parity skipped")
else:
    keyed = """
        select  EventTypeId,
                concat_ws('|', EventTypeId, coalesce(upper(trim(SourcePolicyReference)), ''),
                          coalesce(lower(trim(SourceQuoteReference)), ''), coalesce(trim(EventSourceReference), '')) as RefKey,
                date_trunc('MINUTE', EventDateTime)                    as EventMinute,
                cast(EventDateTime as date)                            as EventDay,
                case when SCV_Customer_Key > 0 then SCV_Customer_Key end as SCV_Customer_Key
        from    {t}
        where   EventDateTime >= timestamp '{s}' and EventDateTime < timestamp '{e}'"""
    for side, t in [("old", old_table), ("new", f"{target_schema}.ScvEventStream")]:
        spark.sql(f"create or replace temporary view parity_{side} as " + keyed.format(t=t, s=start_time, e=end_time))
    spark.sql(f"""
        create or replace table {target_schema}.EventStreamParity as
        select  coalesce(o.EventTypeId, n.EventTypeId)                  as EventTypeId,
                coalesce(o.Events, 0)                                  as OldEvents,
                coalesce(n.Events, 0)                                  as NewEvents,
                coalesce(o.MinuteKeys, 0)                              as OldMinuteKeys,
                coalesce(n.MinuteKeys, 0)                              as NewMinuteKeys,
                coalesce(o.DayKeys, 0)                                 as OldDayKeys,
                coalesce(n.DayKeys, 0)                                 as NewDayKeys,
                coalesce(m.MatchedMinute, 0)                           as MatchedMinute,
                coalesce(d.MatchedDay, 0)                              as MatchedDay,
                coalesce(o.DayKeys, 0) - coalesce(d.MatchedDay, 0)     as OnlyOldDayKeys,
                coalesce(n.DayKeys, 0) - coalesce(d.MatchedDay, 0)     as OnlyNewDayKeys,
                coalesce(m.ScvBothLinked, 0)                           as ScvBothLinked,
                coalesce(m.ScvAgree, 0)                                as ScvAgree,
                coalesce(o.Unlinked, 0)                                as OldUnlinked,
                coalesce(n.Unlinked, 0)                                as NewUnlinked,
                '{batch_id}'                                           as AdsBatchId
        from    (select EventTypeId, count(*) as Events, count(distinct RefKey, EventMinute) as MinuteKeys,
                        count(distinct RefKey, EventDay) as DayKeys,
                        sum(case when SCV_Customer_Key is null then 1 else 0 end) as Unlinked
                 from parity_old group by EventTypeId) o
        full join (select EventTypeId, count(*) as Events, count(distinct RefKey, EventMinute) as MinuteKeys,
                          count(distinct RefKey, EventDay) as DayKeys,
                          sum(case when SCV_Customer_Key is null then 1 else 0 end) as Unlinked
                   from parity_new group by EventTypeId) n
                on n.EventTypeId = o.EventTypeId
        left join (select o.EventTypeId, count(*) as MatchedMinute,
                          sum(case when o.SCV_Customer_Key is not null and n.SCV_Customer_Key is not null then 1 else 0 end) as ScvBothLinked,
                          sum(case when o.SCV_Customer_Key = n.SCV_Customer_Key then 1 else 0 end) as ScvAgree
                   from (select EventTypeId, RefKey, EventMinute, min(SCV_Customer_Key) as SCV_Customer_Key
                         from parity_old group by EventTypeId, RefKey, EventMinute) o
                   join (select EventTypeId, RefKey, EventMinute, min(SCV_Customer_Key) as SCV_Customer_Key
                         from parity_new group by EventTypeId, RefKey, EventMinute) n
                          on n.EventTypeId = o.EventTypeId and n.RefKey = o.RefKey and n.EventMinute = o.EventMinute
                   group by o.EventTypeId) m
                on m.EventTypeId = coalesce(o.EventTypeId, n.EventTypeId)
        left join (select o.EventTypeId, count(*) as MatchedDay
                   from (select distinct EventTypeId, RefKey, EventDay from parity_old) o
                   join (select distinct EventTypeId, RefKey, EventDay from parity_new) n
                          on n.EventTypeId = o.EventTypeId and n.RefKey = o.RefKey and n.EventDay = o.EventDay
                   group by o.EventTypeId) d
                on d.EventTypeId = coalesce(o.EventTypeId, n.EventTypeId)""")
    spark.sql(f"""
        select  *, round(100 * MatchedDay / greatest(OldDayKeys, 1), 1) as PctOldMatchedDay,
                round(100 * ScvAgree / greatest(ScvBothLinked, 1), 1) as PctScvAgree
        from    {target_schema}.EventStreamParity
        order by EventTypeId""").show(1000, truncate=False)


In [ ]:
%%pyspark
ads_run_log("22_Parity", "succeeded")
